In [1]:
# Taakverdeling
# Dennis doet exercise 2 
# Kay doet exercise 1,3.2
# Bas doet exercise 3.1

In [3]:
# Exercise 1 (Kay)

import pandas as pd
import calendar
from datetime import datetime

# Bestand inladen (Kay)
df = pd.read_excel('hotelBookings.xlsx')
df.head(n=5)


# Function for inspecting the data (Kay)
def inspect_data(df: pd.DataFrame) -> None:
    """Inspect the dataset for errors and missing values. Kay"""

    print("Missing values:")
    print(df.isna().sum())

    print("\nDuplicates:")
    print(df.duplicated().sum())

    print("\nData types:")
    print(df.dtypes)


# Function for cleaning text (Kay)
def clean_text(df: pd.DataFrame) -> pd.DataFrame:
    """Remove unnecessary spaces and correct spelling mistakes. Kay"""

    # Step 1: Remove unnecessary spaces
    for col in df.select_dtypes(include=['object', 'string']).columns:
        df[col] = df[col].astype('string').str.strip()

    # Step 2: Fix structural errors
    df['market_segment'] = df['market_segment'].replace({
        'G ROUps': 'Groups'
    })

    df['customer_type'] = df['customer_type'].replace({
        'G ROUp': 'Group'
    })

    # Step 3: Standardize country codes
    df['country'] = df['country'].replace({
        'CN': 'CHN'
    })

    return df


# Function for cleaning numerical data (Kay)
def clean_numbers(df: pd.DataFrame) -> pd.DataFrame:
    """Correct invalid numerical values and outliers. Kay"""

    # Step 1: Convert columns to numeric values
    columns = [
        'adults',
        'children',
        'babies',
        'stays_in_week_nights',
        'stays_in_weekend_nights',
        'adr'
    ]

    for col in columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

        # Step 2: Replace negative values with missing values
        df.loc[df[col] < 0, col] = float('nan')

    # Step 3: Correct an incorrect number of nights
    df.loc[df['stays_in_week_nights'] == 4.3,
           'stays_in_week_nights'] = 4

    # Step 4: Replace unrealistic numbers of guests
    df.loc[df['adults'] > 10, 'adults'] = float('nan')
    df.loc[df['children'] > 10, 'children'] = float('nan')

    return df


# Function for cleaning dates (Kay)
def clean_dates(df: pd.DataFrame) -> pd.DataFrame:
    """Correct incorrect years and recover missing months. Kay"""

    # Step 1: Correct incorrect years
    df['arrival_date_year'] = df['arrival_date_year'].replace({
        2099: 2015
    })

    # Step 2: Create dictionary for months
    months = {
        i: month for i, month in enumerate(calendar.month_name)
        if month
    }

    # Step 3: Recover missing months using week numbers
    for index in df.index[df['arrival_date_month'].isna()]:

        year = df.at[index, 'arrival_date_year']
        week = df.at[index, 'arrival_date_week_number']
        day = df.at[index, 'arrival_date_day_of_month']

        if pd.isna(year) or pd.isna(week) or pd.isna(day):
            continue

        possible_months = []

        for month_number in range(1, 13):
            try:
                date = datetime(int(year), month_number, int(day))

                # Dataset uses Sunday-based week numbers
                if int(date.strftime('%U')) + 1 == int(week):
                    possible_months.append(month_number)

            except ValueError:
                continue

        if len(possible_months) == 1:
            df.at[index, 'arrival_date_month'] = months[possible_months[0]]

    return df


# Main function for data cleaning (Kay)
def data_cleaning1(file_path: str) -> pd.DataFrame:
    """Clean the Hotel Bookings Excel dataset using other functions. Kay"""

    # Step 1: Read the Excel file
    df = pd.read_excel(file_path)

    # Step 2: Validation of data
    if 'hotel' not in df.columns:
        raise ValueError("Column 'hotel' is missing in the dataset.")

    # Step 3: Inspect original data
    inspect_data(df)

    # Step 4: Delete completely empty rows
    df = df.dropna(how='all')

    # Step 5: Clean text
    df = clean_text(df)

    # Step 6: Clean numerical values
    df = clean_numbers(df)

    # Step 7: Clean dates
    df = clean_dates(df)

    # Step 8: Check for remaining errors
    print("\nRemaining missing values:")
    print(df.isna().sum())

    print("\nPossible duplicate bookings:")
    print(df.duplicated().sum())

    return df


# Function for Excel output (Kay)
def cleaned_hotel_data(df_clean: pd.DataFrame) -> None:
    """Save the cleaned hotel data in an Excel file. Kay"""

    df_clean.to_excel("hotelBookings_cleaned.xlsx", index=False)


# Executing the functions (Kay)
cleaned_df = data_cleaning1('hotelBookings_repaired.xlsx')
cleaned_hotel_data(cleaned_df)

display(cleaned_df.head())

BadZipFile: File is not a zip file

In [ ]:
# Exercise 3.1 driedubbele supercode (Bas)

# Bestand inladen (Bas)
df = pd.read_excel('tweets.xlsx')
df.head(n=5)

# Benodigde libraries importeren (Bas)
%pip install langdetect

from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException
from langdetect.lang_detect_exception import LangDetectException

# function for data cleaning (Bas)
def data_cleaning3_1(file_path: str)->dataframe:
    """This data cleaning function clears the tweets Excel file 
    and also provides the wanted output (tweet and language of tweet). Bas"""
    
    # Step 5: validation of data
    df = pd.read_excel(file_path)
    if 'Tweet' not in df.columns:
        raise ValueError("Column 'Tweet' is missing in the dataset.")
    
    # Step 1: Deleting duplicates
    df = df.drop_duplicates(subset=['Tweet'])
    
    # Step 2: Fill in missing tweets
    df['Tweet'] = df['Tweet'].fillna('Unknown')
    
    # Step 3: Fix structural errors (standarize)
    df['Tweet'] = df['Tweet'].astype(str).str.strip()
    
    # Step 4: Filter unwanted 'outliers'
    df = df[df['Tweet'] != '']
    
    # Language detection (using the langedetect library that was imported)
    def detect_lang(text):
        """Automatically determine the language of the tweet using the langedetect library. 
        Output is also put in the required format. Bas"""
        try:
            return detect(text)
        except (LangDetectException, Exception):
            return 'Unknown'
            
    df['Language'] = df['Tweet'].apply(detect_lang)
    df_output = df[['Tweet', 'Language']]
    return df_output

# function for excel output for cleaned data (Bas)
def cleaned_tweet_data(df_clean):
    """Save the cleaned data in an Excel file called cleaned_tweets.xlsx Bas"""
    df_clean.to_excel("cleaned_tweets.xlsx", index=False)

# function for excel output for required output (Bas)
def tweet_required_output(df_output):
    """Save ['Tweet','Language'] is an Excel file called required_output.xlsx Bas"""
    df_output.to_excel("required_output.xlsx",index=False)

# Executing the functions (Bas)
cleaned_df = data_cleaning3_1('tweets.xlsx')
cleaned_tweet_data(cleaned_df)
tweet_required_output(cleaned_df)

In [1]:
# exercise 3.2 (Kay)

# Benodigde libraries importeren
%pip install textblob nltk

import pandas as pd
import nltk
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer

# Download necessary data for sentiment analysis
nltk.download('vader_lexicon', quiet=True)

analyzer = SentimentIntensityAnalyzer()


# Function for English sentiment analysis
def analyze_sentiment_english(tweet: str) -> str:
    """Analyze the sentiment of English tweets using TextBlob."""

    # Step 1: Calculate polarity score
    polarity = TextBlob(tweet).sentiment.polarity

    # Step 2: Determine sentiment
    if polarity > 0:
        return 'positive'
    elif polarity < 0:
        return 'negative'
    else:
        return 'neutral'


# Function for other language sentiment analysis
def analyze_sentiment_other(tweet: str) -> str:
    """Analyze the sentiment of non-English tweets using NLTK."""

    # Step 1: Calculate compound score
    compound = analyzer.polarity_scores(tweet)['compound']

    # Step 2: Determine sentiment
    if compound >= 0.05:
        return 'positive'
    elif compound <= -0.05:
        return 'negative'
    else:
        return 'neutral'


# Main function for sentiment detection
def sentiment_detection3_2(file_path: str) -> pd.DataFrame:
    """Perform sentiment analysis based on the language of each tweet."""

    # Step 1: Read the Excel file
    df = pd.read_excel(file_path)

    # Step 2: Validation of data
    if 'Tweet' not in df.columns:
        raise ValueError("Column 'Tweet' is missing in the dataset.")

    if 'Language' not in df.columns:
        raise ValueError("Column 'Language' is missing in the dataset.")

    # Step 3: Fill in missing values
    df['Tweet'] = df['Tweet'].fillna('').astype(str).str.strip()
    df['Language'] = df['Language'].fillna('Unknown')

    # Step 4: Analyze sentiment based on language
    sentiments = []

    for tweet, language in zip(df['Tweet'], df['Language']):

        if tweet == '' or tweet.lower() == 'unknown':
            sentiment = 'neutral'

        elif str(language).strip().lower() == 'en':
            sentiment = analyze_sentiment_english(tweet)

        else:
            sentiment = analyze_sentiment_other(tweet)

        sentiments.append(sentiment)

    # Step 5: Add sentiment column
    df['Sentiment'] = sentiments

    # Step 6: Select required output
    df_output = df[['Tweet', 'Language', 'Sentiment']]

    return df_output


# Function for Excel output
def sentiment_required_output(df_output: pd.DataFrame) -> None:
    """Save the sentiment analysis results in an Excel file."""

    df_output.to_excel("sentiment_output.xlsx", index=False)


# Executing the functions
sentiment_df = sentiment_detection3_2('required_output.xlsx')
sentiment_required_output(sentiment_df)

display(sentiment_df.head())

Note: you may need to restart the kernel to use updated packages.


,Tweet,Language,Sentiment
0,BugÃ¼n bulusmami lazimdiii,tr,neutral
1,Volkan konak adami tribe sokar yemin ederim :D,tr,positive
2,Bed,de,neutral
3,I felt my first flash of violence at some fool...,en,positive
4,Ladies drink and get in free till 10:30,af,positive
